# DPAD Forecast Sweep

Runs `--phases forecasts` for DPAD e1000 configs on Kaggle GPU.

Uses conda Python 3.11 + TF 2.15.1 to match local training env.

Dataset: `giedriusmirklys/dpad-splits`

In [ ]:
SESSIONS=['PDI1_S2','PDI1_S4','PDI4_S2','PDI4_S3']
MODES=['z-as-behavior','z-as-neural']
H_VALUES=[4.0, 5.0]

In [ ]:
import subprocess, sys, os
from pathlib import Path

CONDA_DIR = Path('/kaggle/working/conda')
PY = str(CONDA_DIR / 'envs/neuro/bin/python')

def run(*cmd, **kw):
    print('$', ' '.join(str(c) for c in cmd))
    r = subprocess.run(list(cmd), capture_output=True, text=True, **kw)
    if r.stdout: print(r.stdout[-800:])
    if r.returncode != 0: print('STDERR:', r.stderr[-800:])
    return r.returncode

if not Path(PY).exists():
    run('wget', '-q',
        'https://repo.anaconda.com/miniconda/Miniconda3-py311_23.11.0-2-Linux-x86_64.sh',
        '-O', '/tmp/miniconda.sh')
    run('bash', '/tmp/miniconda.sh', '-b', '-p', str(CONDA_DIR))
    CONDA = str(CONDA_DIR / 'bin/conda')
    run(CONDA, 'create', '-n', 'neuro', 'python=3.11', '-y', '-q')
    run(PY, '-m', 'pip', 'install', '-q', 'tensorflow==2.15.1')
    run(PY, '-m', 'pip', 'install', '-q', '--no-deps', '--ignore-requires-python',
        'DPAD==0.0.9', 'PSID==1.2.6')
    run(PY, '-m', 'pip', 'install', '-q', 'polars>=1.0.0', 'pyyaml', 'pyarrow',
        'pandas', 'scikit-learn', 'scipy', 'numpy', 'mne', 'h5py', 'torch', 'statsmodels')
    print('env ready')
else:
    print('env already exists:', PY)

In [ ]:
import os
from pathlib import Path

WORK  = Path('/kaggle/working')
DATA  = Path('/kaggle/input/datasets/giedriusmirklys/dpad-splits')
if not DATA.exists():
    raise SystemExit(f'no dpad-splits mount at {DATA}')
print('DATA mount:', DATA)

# symlink result dirs
(WORK/'results'/'dpad').mkdir(parents=True, exist_ok=True)
for sv in sorted(DATA.glob('results/dpad/dpad_*_dbs_*')):
    dst = WORK/'results'/'dpad'/sv.name
    dst.mkdir(parents=True, exist_ok=True)
    for item in sv.iterdir():
        l = dst/item.name
        if not l.exists(): os.symlink(item, l)
    (dst/'forecast').mkdir(exist_ok=True)

# YAML configs
dst_setups = WORK/'training'/'setups'/'dpad_modal'
dst_setups.mkdir(parents=True, exist_ok=True)
for y in (DATA/'training/setups/dpad_modal').glob('*.yaml'):
    l = dst_setups/y.name
    if not l.exists(): os.symlink(y, l)

(WORK/'logs'/'dpad').mkdir(parents=True, exist_ok=True)
(WORK/'tmp_cfgs').mkdir(exist_ok=True)

staged = len(list((WORK/'results'/'dpad').iterdir()))
yamls  = len(list(dst_setups.glob('*.yaml')))
print(f'staged {staged} variant dirs, {yamls} yamls')

In [ ]:
import re, subprocess, textwrap
from pathlib import Path

WORK = Path('/kaggle/working')
PY   = str(Path('/kaggle/working/conda/envs/neuro/bin/python'))
SRC  = str(Path('/kaggle/input/datasets/giedriusmirklys/dpad-splits/src'))

SCRIPT = WORK / 'tmp_cfgs' / 'run_forecast.py'
SCRIPT.write_text(textwrap.dedent(f'''
    import re, sys, traceback
    from pathlib import Path
    sys.path.insert(0, {repr(SRC)})
    import os; os.chdir({repr(str(WORK))})
    import tensorflow as tf; print('GPU:', tf.config.list_physical_devices('GPU'))
    from utils.config import get_config
    from utils.logger import setup_logging
    from training.pipelines._base import FrameworkPipeline

    YAML_DIR = Path({repr(str(WORK/'training'/'setups'/'dpad_modal'))})
    TMP      = Path({repr(str(WORK/'tmp_cfgs'))})
    SESSIONS = {repr(SESSIONS)}
    MODES    = {repr(MODES)}
    H_VALUES = {repr(H_VALUES)}

    _H_RE = re.compile(r\"( {{4}}h_grid:\\n)(?:    - \\S+\\n)+\")
    def patch(text, hv):
        blk = \"    h_grid:\\n\" + \"\".join(f\"    - {{h}}\\n\" for h in hv)
        return _H_RE.sub(blk, text)

    errors = []
    for mode in MODES:
        for s in SESSIONS:
            yp = YAML_DIR / f\"dpad_modal_{{s}}_{{mode}}.yaml\"
            if not yp.exists(): print(f\"SKIP {{yp.name}}\"); continue
            print(f\"\\n==== {{s}} {{mode}} h= {{H_VALUES}} ====\")
            tmp = TMP / f\"dpad_modal_{{s}}_{{mode}}_hpatch.yaml\"
            tmp.write_text(patch(yp.read_text(), H_VALUES))
            try:
                config = get_config(str(tmp))
                log = setup_logging(f\"dpad_{{s}}_{{mode}}\",
                    Path({repr(str(WORK))}) / \"logs\" / \"dpad\" / f\"{{s}}_{{mode}}.log\")
                FrameworkPipeline(config, log, phases=(\"forecasts\",)).run()
                print(f\"DONE {{s}}/{{mode}}\")
            except Exception as exc:
                traceback.print_exc()
                errors.append((s, mode, str(exc)))
            finally:
                tmp.unlink(missing_ok=True)

    print(f\"\\nFinished. {{len(errors)}} errors.\")
    for s, m, e in errors: print(f\"  FAILED {{s}}/{{m}}: {{e}}\")
''').strip())

import os as _os
env = {
    **_os.environ,
    'MPLBACKEND': 'Agg',
    'LD_LIBRARY_PATH': '/usr/local/cuda/lib64:' + _os.environ.get('LD_LIBRARY_PATH', ''),
}
result = subprocess.run(
    [PY, str(SCRIPT)],
    capture_output=False,
    text=True,
    env=env
)
print('exit code:', result.returncode)

In [ ]:
from pathlib import Path
WORK = Path('/kaggle/working')
for h in H_VALUES:
    hd = f'h{h:g}'
    print('===', hd, '===')
    for p in sorted(WORK.glob(f'results/dpad/dpad_*_dbs_both/forecast/{hd}/test/*.parquet')):
        print(' ', p.parent.parent.parent.parent.name, '- test OK')